## 股息单因子回测

预期股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有announce_date在这之间的总分红；

~~静态股息率：上一财年总div_pre_tax和 / **不复权**市场价格，计入ex_date在当前日期之前且div_year为当前日期上一年的总分红；~~

动态股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有ex_date在这之间的总分红；

~~div_after_tax~~ div_pre_tax ：税前股息反映的是公司真实分红能力，税后股息只反映单个投资者的到手收益，前者更适合做因子/估值。

Expected Dividend Yield (TTM):
Sum of div_pre_tax over the most recent 365 calendar days / non-adjusted market price.
Includes total dividends whose announce_date falls within the date range.
Static Dividend Yield:
Total div_pre_tax of the previous fiscal year / non-adjusted market price.
Includes total dividends whose ex_date is before the current date and div_year equals the year prior to the current date.
**Dynamic Dividend Yield (TTM)**:
Sum of div_pre_tax over the most recent 365 calendar days / non-adjusted market price.
Includes total dividends whose ex_date falls within the date range.
div_pre_tax (not div_after_tax):
Pre-tax dividend reflects a company’s true profitability and cash distribution capacity,
while after-tax dividend only represents individual investors’ actual received income.
The former is more suitable for factor construction and valuation analysis.

In [47]:
import pandas as pd
div_factor_dt=pd.read_parquet("data/dividend_factor.parquet")
div_factor_dt.head()

,stock_code,trade_date,expected_div_yield,static_div_yield,dynamic_div_yield,year,industry_name
0,000001.SZ,2013-01-04,0.0,0.0,0.0,2013,银行
1,000001.SZ,2013-01-07,0.0,0.0,0.0,2013,银行
2,000001.SZ,2013-01-08,0.0,0.0,0.0,2013,银行
3,000001.SZ,2013-01-09,0.0,0.0,0.0,2013,银行
4,000001.SZ,2013-01-10,0.0,0.0,0.0,2013,银行


In [ ]:
temp=pd.read_parquet("data/ref_data/ETF_hold_510300.SH.parquet")
temp=temp[temp['end_date']=='2018-06-30']
temp.head()

,fund_code,end_date,stock_code,index_value,report_year,report_type


由于给定的数据中缺少早期数据，采用的回测方法是早期全样本，后期沪深300.因为沪深300股票数量较少，采用每个行业TOP1/2股票

In [55]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from scipy.stats.mstats import winsorize

CONFIG = {
    "TOP_N": 2,
    "BOTTOM_N": 2,
    "FACTOR_COLUMN": "dynamic_div_yield",
    "USE_WINSORIZE": False,
    "WINSORIZE_LIMITS": (0.01, 0.01),
    "STANDARDIZE_BY_INDUSTRY": False,
    "DIVIDEND_FACTOR_FILE": 'data/dividend_factor.parquet',
    "FUND_FILE": 'data/ref_data/ETF_hold_510300.SH.parquet',
    "SIGNAL_SAVE_PATH": 'records/industry_selection.parquet'
}

def generate_industry_signals(config):
    os.makedirs(os.path.dirname(config['SIGNAL_SAVE_PATH']), exist_ok=True)
    print("=" * 60)
    print("行业分层选股信号生成 (每个行业 Top{} / Bottom{})".format(config['TOP_N'], config['BOTTOM_N']))
    print(f"因子列: {config['FACTOR_COLUMN']}")
    print(f"行业内标准化: {'开启' if config['STANDARDIZE_BY_INDUSTRY'] else '关闭'}")
    print("=" * 60)

    # ----------------------------
    # 1. 加载数据
    # ----------------------------
    dividend_factor = pd.read_parquet(config["DIVIDEND_FACTOR_FILE"])
    fund_df = pd.read_parquet(config["FUND_FILE"])

    # 剔除行业名缺失的行
    dividend_factor = dividend_factor.dropna(subset=['industry_name'])

    # 统一日期格式
    dividend_factor['trade_date'] = pd.to_datetime(dividend_factor['trade_date'])
    fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})
    fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

    # 判断是否需要成分股过滤
    if fund_df.empty:
        print("⚠️ 警告：基金持仓文件为空，将使用全部股票（不进行成分股过滤）")
        use_all_stocks = True
        fund_holdings = pd.DataFrame()  # 占位
    else:
        use_all_stocks = False
        def get_position_valid_period(row):
            year = row['report_year']
            if row['report_type'] == '中报':
                start_date = pd.to_datetime(f'{year}-01-01')
                end_date = pd.to_datetime(f'{year}-06-30')
            elif row['report_type'] == '年报':
                start_date = pd.to_datetime(f'{year}-07-01')
                end_date = pd.to_datetime(f'{year}-12-31')
            else:
                start_date = end_date = row['report_end_date']
            return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

        fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)
        fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

        # 过滤出历史上所有出现过的沪深300成分股（初步筛选）
        csi_300_stocks = fund_holdings['stock_code'].unique()
        dividend_factor = dividend_factor[dividend_factor['stock_code'].isin(csi_300_stocks)]

    # 检查因子列
    factor_col = config["FACTOR_COLUMN"]
    if factor_col not in dividend_factor.columns:
        raise KeyError(f"因子文件缺少列 '{factor_col}'，可用列：{dividend_factor.columns.tolist()}")

    # ----------------------------
    # 2. 缩尾处理（可选）
    # ----------------------------
    if config['USE_WINSORIZE']:
        print("执行因子缩尾处理...")
        dividend_factor[factor_col] = winsorize(
            dividend_factor[factor_col].values,
            limits=config['WINSORIZE_LIMITS'],
            inclusive=(True, True)
        )

    # ----------------------------
    # 3. 行业内标准化（可选）
    # ----------------------------
    if config['STANDARDIZE_BY_INDUSTRY']:
        print("计算行业内截面标准化因子...")
        dividend_factor['factor_std'] = dividend_factor.groupby(['trade_date', 'industry_name'])[factor_col].transform(
            lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
        )
        score_col = 'factor_std'
    else:
        score_col = factor_col

    # ----------------------------
    # 4. 按日、按行业生成信号
    # ----------------------------
    all_signals = []
    unique_dates = sorted(dividend_factor['trade_date'].unique())

    for trade_date in tqdm(unique_dates, desc="处理交易日"):
        daily = dividend_factor[dividend_factor['trade_date'] == trade_date].copy()

        if daily.empty:
            continue

        # ---------- 获取当日有效成分股 ----------
        if use_all_stocks:
            valid_codes = daily['stock_code'].unique()
        else:
            valid_codes = fund_holdings[
                (fund_holdings['pos_start_date'] <= trade_date) &
                (fund_holdings['pos_end_date'] >= trade_date)
            ]['stock_code'].tolist()
            if len(valid_codes) == 0:
                valid_codes = daily['stock_code'].unique()

        daily = daily[daily['stock_code'].isin(valid_codes)]
        daily = daily.dropna(subset=[score_col])

        if daily.empty:
            continue

        daily['signal'] = 'none'

        for industry, group in daily.groupby('industry_name'):
            if len(group) == 0:
                continue

            group_sorted = group.sort_values(score_col, ascending=False)
            n_stocks = len(group_sorted)
            top_n = min(config['TOP_N'], n_stocks)
            bottom_n = min(config['BOTTOM_N'], n_stocks)

            indices = group_sorted.index

            if top_n > 0:
                long_indices = indices[:top_n]
                daily.loc[long_indices, 'signal'] = 'long'

            remaining_indices = indices[top_n:]
            if bottom_n > 0 and len(remaining_indices) > 0:
                short_indices = remaining_indices[-min(bottom_n, len(remaining_indices)):]
                daily.loc[short_indices, 'signal'] = 'short'

        result = daily[['trade_date', 'stock_code', 'industry_name', score_col, 'signal']].copy()
        result.rename(columns={score_col: 'factor_value'}, inplace=True)
        all_signals.append(result)

    if not all_signals:
        raise ValueError("没有生成任何信号，请检查数据")

    signal_df = pd.concat(all_signals, ignore_index=True)

    # ----------------------------
    # 5. 保存结果
    # ----------------------------
    signal_df.to_parquet(config['SIGNAL_SAVE_PATH'], index=False)
    print(f"\n信号文件已保存至：{config['SIGNAL_SAVE_PATH']}")
    print(f"共 {signal_df['trade_date'].nunique()} 个交易日，{len(signal_df)} 条记录")
    print("\n信号分布（总体）：")
    print(signal_df['signal'].value_counts())
    print("\n各行业信号分布示例（前5个行业）：")
    industry_stats = signal_df.groupby(['industry_name', 'signal']).size().unstack(fill_value=0)
    print(industry_stats.head())

    return signal_df

if __name__ == "__main__":
    signals = generate_industry_signals(CONFIG)

行业分层选股信号生成 (每个行业 Top2 / Bottom2)
因子列: dynamic_div_yield
行业内标准化: 关闭


处理交易日: 100%|██████████| 3134/3134 [01:25<00:00, 36.49it/s]



信号文件已保存至：records/industry_selection.parquet
共 3134 个交易日，1299261 条记录

信号分布（总体）：
signal
none     950023
long     181958
short    167280
Name: count, dtype: int64

各行业信号分布示例（前5个行业）：
signal         long    none  short
industry_name                     
交通运输           6268   41377   6268
传媒             6268   16834   5909
公用事业           6268   34497   6268
农林牧渔           6268   15638   6268
医药生物           6268  105126   6268


In [65]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2016-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2025-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 437

=== 调仓日期：2016-07-01 ===
目标股票数：59 | 总资产：1000000.00 | 每只目标市值：16949.15
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2016-07-08 ===
目标股票数：59 | 总资产：1027907.68 | 每只目标市值：17422.16
实际交易总额：299696.94 | 手续费：299.70 | 剩余现金：-299.70

=== 调仓日期：2016-07-15 ===
目标股票数：59 | 总资产：1059743.71 | 每只目标市值：17961.76
实际交易总额：166405.26 | 手续费：166.41 | 剩余现金：-166.41

=== 调仓日期：2016-07-22 ===
目标股票数：59 | 总资产：1044936.05 | 每只目标市值：17710.78
实际交易总额：262984.68 | 手续费：262.98 | 剩余现金：-262.98

=== 调仓日期：2016-07-29 ===
目标股票数：59 | 总资产：1042591.97 | 每只目标市值：17671.05
实际交易总额：313521.48 | 手续费：313.52 | 剩余现金：-313.52

=== 调仓日期：2016-08-05 ===
目标股票数：59 | 总资产：1038987.92 | 每只目标市值：17609.96
实际交易总额：123602.05 | 手续费：123.60 | 剩余现金：-123.60

=== 调仓日期：2016-08-12 ===
目标股票数：59 | 总资产：1064450.81 | 每只目标市值：18041.54
实际交易总额：166490.69 | 手续费：166.49 | 剩余现金：-166.49

=== 调仓日期：2016-08-19 ===
目标股票数：59 | 总资产：1101535.88 | 每只目标市值：18670.10
实际交易总额：220248.82 | 手续费：220.25 | 剩余现金：-220.25

=== 调仓日期：2016-08-26 ===
目标股票数：59 | 总资

## 单因子回测稳健性，检验不同时间子样本的策略表现

In [58]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2016-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2018-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 98

=== 调仓日期：2016-07-01 ===
目标股票数：59 | 总资产：1000000.00 | 每只目标市值：16949.15
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2016-07-08 ===
目标股票数：59 | 总资产：1027907.68 | 每只目标市值：17422.16
实际交易总额：299696.94 | 手续费：299.70 | 剩余现金：-299.70

=== 调仓日期：2016-07-15 ===
目标股票数：59 | 总资产：1059743.71 | 每只目标市值：17961.76
实际交易总额：166405.26 | 手续费：166.41 | 剩余现金：-166.41

=== 调仓日期：2016-07-22 ===
目标股票数：59 | 总资产：1044936.05 | 每只目标市值：17710.78
实际交易总额：262984.68 | 手续费：262.98 | 剩余现金：-262.98

=== 调仓日期：2016-07-29 ===
目标股票数：59 | 总资产：1042591.97 | 每只目标市值：17671.05
实际交易总额：313521.48 | 手续费：313.52 | 剩余现金：-313.52

=== 调仓日期：2016-08-05 ===
目标股票数：59 | 总资产：1038987.92 | 每只目标市值：17609.96
实际交易总额：123602.05 | 手续费：123.60 | 剩余现金：-123.60

=== 调仓日期：2016-08-12 ===
目标股票数：59 | 总资产：1064450.81 | 每只目标市值：18041.54
实际交易总额：166490.69 | 手续费：166.49 | 剩余现金：-166.49

=== 调仓日期：2016-08-19 ===
目标股票数：59 | 总资产：1101535.88 | 每只目标市值：18670.10
实际交易总额：220248.82 | 手续费：220.25 | 剩余现金：-220.25

=== 调仓日期：2016-08-26 ===
目标股票数：59 | 总资产

In [61]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2018-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2020-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 97

=== 调仓日期：2018-07-02 ===
目标股票数：60 | 总资产：1000000.00 | 每只目标市值：16666.67
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2018-07-09 ===
目标股票数：60 | 总资产：1008712.09 | 每只目标市值：16811.87
实际交易总额：393457.02 | 手续费：393.46 | 剩余现金：-393.46

=== 调仓日期：2018-07-16 ===
目标股票数：60 | 总资产：1020332.16 | 每只目标市值：17005.54
实际交易总额：295677.28 | 手续费：295.68 | 剩余现金：-295.68

=== 调仓日期：2018-07-23 ===
目标股票数：60 | 总资产：1038792.15 | 每只目标市值：17313.20
实际交易总额：539225.77 | 手续费：539.23 | 剩余现金：-539.23

=== 调仓日期：2018-07-30 ===
目标股票数：60 | 总资产：1043975.78 | 每只目标市值：17399.60
实际交易总额：232766.53 | 手续费：232.77 | 剩余现金：-232.77

=== 调仓日期：2018-08-06 ===
目标股票数：60 | 总资产：971105.00 | 每只目标市值：16185.08
实际交易总额：223258.67 | 手续费：223.26 | 剩余现金：-223.26

=== 调仓日期：2018-08-13 ===
目标股票数：60 | 总资产：1011052.61 | 每只目标市值：16850.88
实际交易总额：291837.48 | 手续费：291.84 | 剩余现金：-291.84

=== 调仓日期：2018-08-20 ===
目标股票数：60 | 总资产：971845.25 | 每只目标市值：16197.42
实际交易总额：181866.32 | 手续费：181.87 | 剩余现金：-181.87

=== 调仓日期：2018-08-27 ===
目标股票数：60 | 总资产：1

In [62]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2020-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2022-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 98

=== 调仓日期：2020-07-01 ===
目标股票数：61 | 总资产：1000000.00 | 每只目标市值：16393.44
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2020-07-08 ===
目标股票数：61 | 总资产：1109586.12 | 每只目标市值：18189.94
实际交易总额：346357.50 | 手续费：346.36 | 剩余现金：-346.36

=== 调仓日期：2020-07-15 ===
目标股票数：61 | 总资产：1117171.89 | 每只目标市值：18314.29
实际交易总额：436729.36 | 手续费：436.73 | 剩余现金：-436.73

=== 调仓日期：2020-07-22 ===
目标股票数：61 | 总资产：1124633.85 | 每只目标市值：18436.62
实际交易总额：291207.15 | 手续费：291.21 | 剩余现金：-291.21

=== 调仓日期：2020-07-29 ===
目标股票数：61 | 总资产：1114607.25 | 每只目标市值：18272.25
实际交易总额：192992.77 | 手续费：192.99 | 剩余现金：-192.99

=== 调仓日期：2020-08-05 ===
目标股票数：61 | 总资产：1153932.84 | 每只目标市值：18916.93
实际交易总额：271081.40 | 手续费：271.08 | 剩余现金：-271.08

=== 调仓日期：2020-08-12 ===
目标股票数：61 | 总资产：1134763.82 | 每只目标市值：18602.69
实际交易总额：261844.64 | 手续费：261.84 | 剩余现金：-261.84

=== 调仓日期：2020-08-19 ===
目标股票数：61 | 总资产：1162046.99 | 每只目标市值：19049.95
实际交易总额：178998.75 | 手续费：179.00 | 剩余现金：-179.00

=== 调仓日期：2020-08-26 ===
目标股票数：61 | 总资产

In [63]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2022-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2024-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 97

=== 调仓日期：2022-07-01 ===
目标股票数：55 | 总资产：1000000.00 | 每只目标市值：18181.82
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2022-07-08 ===
目标股票数：55 | 总资产：992318.83 | 每只目标市值：18042.16
实际交易总额：135534.29 | 手续费：135.53 | 剩余现金：-135.53

=== 调仓日期：2022-07-15 ===
目标股票数：55 | 总资产：957263.10 | 每只目标市值：17404.78
实际交易总额：332530.74 | 手续费：332.53 | 剩余现金：-332.53

=== 调仓日期：2022-07-22 ===
目标股票数：55 | 总资产：964348.73 | 每只目标市值：17533.61
实际交易总额：235683.68 | 手续费：235.68 | 剩余现金：-235.68

=== 调仓日期：2022-07-29 ===
目标股票数：55 | 总资产：952924.72 | 每只目标市值：17325.90
实际交易总额：263350.60 | 手续费：263.35 | 剩余现金：-263.35

=== 调仓日期：2022-08-05 ===
目标股票数：55 | 总资产：941138.42 | 每只目标市值：17111.61
实际交易总额：156946.38 | 手续费：156.95 | 剩余现金：-156.95

=== 调仓日期：2022-08-12 ===
目标股票数：55 | 总资产：957404.40 | 每只目标市值：17407.35
实际交易总额：54203.88 | 手续费：54.20 | 剩余现金：-54.20

=== 调仓日期：2022-08-19 ===
目标股票数：55 | 总资产：954771.27 | 每只目标市值：17359.48
实际交易总额：203342.45 | 手续费：203.34 | 剩余现金：-203.34

=== 调仓日期：2022-08-26 ===
目标股票数：55 | 总资产：953524.46

In [64]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 红利策略 全多头回测配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2024-07-01",      # 【可调节】回测开始日期
    "END_DATE": "2025-06-30",        # 【可调节】回测结束日期
    "REBALANCE_FREQ": 5,             # 【可调节】调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件路径
    "SELECTION_FILE": "records/industry_selection.parquet", # 红利选股文件
    "TRANSACTION_FEE": 0.001         # 【可调节】交易手续费（0.1%）
}

# ===================== 执行红利策略回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出绩效指标 =====================
print("\n==================== 沪深300红利策略 绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ===================== 绘制回测图表 =====================
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'沪深300红利策略 | 每{CONFIG["REBALANCE_FREQ"]}日调仓 | 手续费{CONFIG["TRANSACTION_FEE"]*100:.1f}%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 49

=== 调仓日期：2024-07-01 ===
目标股票数：54 | 总资产：1000000.00 | 每只目标市值：18518.52
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2024-07-08 ===
目标股票数：54 | 总资产：968260.56 | 每只目标市值：17930.75
实际交易总额：137069.14 | 手续费：137.07 | 剩余现金：-137.07

=== 调仓日期：2024-07-15 ===
目标股票数：54 | 总资产：983178.41 | 每只目标市值：18207.01
实际交易总额：276831.60 | 手续费：276.83 | 剩余现金：-276.83

=== 调仓日期：2024-07-22 ===
目标股票数：54 | 总资产：985736.29 | 每只目标市值：18254.38
实际交易总额：388143.50 | 手续费：388.14 | 剩余现金：-388.14

=== 调仓日期：2024-07-29 ===
目标股票数：54 | 总资产：955910.57 | 每只目标市值：17702.05
实际交易总额：237487.23 | 手续费：237.49 | 剩余现金：-237.49

=== 调仓日期：2024-08-05 ===
目标股票数：54 | 总资产：944737.53 | 每只目标市值：17495.14
实际交易总额：231123.76 | 手续费：231.12 | 剩余现金：-231.12

=== 调仓日期：2024-08-12 ===
目标股票数：54 | 总资产：948555.24 | 每只目标市值：17565.84
实际交易总额：92013.98 | 手续费：92.01 | 剩余现金：-92.01

=== 调仓日期：2024-08-19 ===
目标股票数：54 | 总资产：942969.86 | 每只目标市值：17462.40
实际交易总额：163047.40 | 手续费：163.05 | 剩余现金：-163.05

=== 调仓日期：2024-08-26 ===
目标股票数：54 | 总资产：924734.01